# AGH-Former v2 - Google Colab Pro
Sade 2 aşamalı yapı: arşivlenmiş vNext global aşaması + tek All23-MVSC yerel düzeltici.
Sırayla çalıştırın: kurulum → smoke → 5 fold validasyon → (kapı geçerse) final test.

Gerekli: `agh_vnext_runs/publication_cv_seed42` (vNext arşivi) ve `all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42` (ön işleme) Drive'da olmalı.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess
CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(CODE_ROOT / 'agh_former_vnext_orthodontic_comparison/requirements.txt')], check=True)

In [ ]:
STAGE_A = Path('/content/drive/MyDrive/orthodontic/agh_vnext_runs/publication_cv_seed42')
PRE = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42')
for fold in range(1, 6):
    assert (STAGE_A / f'fold_{fold}/predictions_val.csv').exists(), fold
    assert (PRE / f'fold_{fold}').exists(), fold
print('Arşiv ve ön işleme bulundu')

## 1. Smoke (Fold 1, 2 epoch, yalnızca validasyon)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_smoke --seed 42

## 2. Beş fold validasyon
Test dosyası açılmaz. Sonunda `validation_gate.json` yazılır.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_val_all_folds --seed 42

In [ ]:
import json
gate = json.loads(Path('/content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42/validation_gate.json').read_text())
print(json.dumps({k: gate[k] for k in ('v2_validation', 'vnext_validation', 'checks', 'run_final_test')}, indent=2))

## 2b. Kapı yeniden hesaplama + tanı raporu (GPU gerekmez)
Kapıyı (fold, kişi) bazında yeniden hesaplar ve `validation_diagnostics/diagnostics_val.md` yazar. Test dosyası okunmaz.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_diagnostics --seed 42

In [ ]:
from IPython.display import Markdown
Markdown(Path('/content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42/validation_diagnostics/diagnostics_val.md').read_text())

## 3. Final test (yalnızca kapı geçtiyse, bir kez)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_final_test --seed 42

In [ ]:
%cd /content/comparative-study
!python -m shared_metrics.aggregate_cv_results --run-dir /content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42 --model 'AGH-Former v2'